# Transversal Clifford gates

This notebook shows how to find the SWAP-transversal logical Clifford gates of a code with `qldpc.circuits.get_transversal_ops`, and how to find a physical circuit that implements a desired logical Clifford operation with `qldpc.circuits.get_transversal_circuit`.  These are constructed via the code-automorphism method of [arXiv:2409.18175](https://arxiv.org/abs/2409.18175).

### NOTICE:
The examples here work out of the box because `qLDPC` has a few groups hard-coded in `qldpc.external.groups`.

Finding transversal gates of other codes requires installing [GAP](https://www.gap-system.org/)/[GUAVA](https://www.gap-system.org/Packages/guava.html).

The methods in this notebook rely on an exponential-time subroutine, namely finding the automorphism group of a classical code (the group of permutations that preserves the code space).  These methods are therefore only practical for small-to-medium-sized codes.

In [1]:
%%capture
%pip install qldpc

In [2]:
import stim

from qldpc import circuits, codes

### The five-qubit code has two SWAP-transversal Clifford operations

... namely, the logical H and S gates, which we can identify by their stabilizer tableaus.

Stabilizer tableaus uniquely characterize a Clifford operation by how it transforms Pauli strings.  See Section 2.3 of [the Stim paper](https://quantum-journal.org/papers/q-2021-07-06-497/pdf/) for more information.

In [3]:
code = codes.FiveQubitCode()
print(code.get_code_params())

transversal_ops = circuits.get_transversal_ops(code)
for idx, (tableau, circuit) in enumerate(transversal_ops, start=1):
    print()
    print("-" * 20)
    print(f"operation #{idx}:")
    print()
    print("tableau:")
    print(tableau)
    print()
    print("circuit:")
    print(circuit)

(5, 1, 3)

--------------------
operation #1:

tableau:
+-xz-
| +-
| ZX

circuit:
X 0
Y 1 2 4
S 0 2
H_YZ 1
H 3 4
SWAP 3 4

--------------------
operation #2:

tableau:
+-xz-
| ++
| YZ

circuit:
S 0 1 2 3 4
SWAP 1 3 1 4 1 2


### The 2x2 toric code has four SWAP-transversal Clifford operations

We can find these using MAGMA by passing `with_magma=True`.

In [4]:
code = codes.ToricCode(2)
print(code.get_code_params())

print()
transversal_ops = circuits.get_transversal_ops(code, with_magma=True)
for idx, (tableau, circuit) in enumerate(transversal_ops, start=1):
    print()
    print("-" * 20)
    print(f"operation #{idx}:")
    print()
    print("tableau:")
    print(tableau)
    print()
    print("circuit:")
    print(circuit)

(4, 2, 2)

Run the following command in MAGMA:

AutomorphismGroup(LinearCode(Matrix(GF(2),2,12,[[1,1,1,1,0,0,0,0,1,1,1,1],[0,0,0,0,1,1,1,1,1,1,1,1]])));

NOTICE: group found in the local MAGMA group cache.  Retrieved group generators (in cycle notation):
[(9, 10)]
[(0, 5), (1, 6), (2, 7), (3, 4)]
[(0, 1)]
[(0, 10), (1, 8), (2, 11), (3, 9)]
[(5, 6)]
[(8, 10)]
[(1, 2)]
[(0, 3)]
[(4, 5)]
[(6, 7)]
[(8, 11)]

If you think that the cached result is incorrect, you can remove it from the cache by running the following commands:

import qldpc
qldpc.cache.clear_entry("magma_groups", """AutomorphismGroup(LinearCode(Matrix(GF(2),2,12,[[1,1,1,1,0,0,0,0,1,1,1,1],[0,0,0,0,1,1,1,1,1,1,1,1]])));""")


--------------------
operation #1:

tableau:
+-xz-xz-
| ++ ++
| __ XZ
| XZ __

circuit:
SWAP 2 3

--------------------
operation #2:

tableau:
+-xz-xz-
| ++ ++
| Z_ ZX
| ZX _X

circuit:
H 0 1 2 3
SWAP 0 2

--------------------
operation #3:

tableau:
+-xz-xz-
| +- +-
| _X XZ
| XZ _X

circuit:
H_YZ 0 1 2 3

### Find the physical circuit that implements a Hadamard on both logical qubits of the 2x2 toric code

In [5]:
print(circuits.get_transversal_circuit(code, stim.Circuit("H 0 1")))

H 0 1 2 3
SWAP 0 1
